In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

BATCH_SIZE = 256
EPOCHS = 50
LR = 3e-4
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

DATA_DIR = "/content"

X_train = pd.read_csv(f"{DATA_DIR}/X_train.csv")
Y_train = pd.read_csv(f"{DATA_DIR}/Y_train.csv")
X_val   = pd.read_csv(f"{DATA_DIR}/X_val.csv")
Y_val   = pd.read_csv(f"{DATA_DIR}/Y_val.csv")
X_test  = pd.read_csv(f"{DATA_DIR}/X_test.csv")

# ─────────────────────────────────────────────
# FIX 1: Build vocab from ALL splits (train+val+test)
# so UNK tokens are minimized
# ─────────────────────────────────────────────
feature_cols = [c for c in X_train.columns if c.startswith("feature_")]
SEQ_LEN = len(feature_cols)

all_values = pd.concat([
    X_train[feature_cols],
    X_val[feature_cols],
    X_test[feature_cols]
]).values.flatten()

unique_actions = np.unique(all_values[~pd.isna(all_values)])
unique_actions = unique_actions[unique_actions != 0]

# idx 0 = PAD, idx 1 = UNK
action2idx = {a: i + 2 for i, a in enumerate(unique_actions)}
action2idx[0]     = 0
action2idx["UNK"] = 1
VOCAB_SIZE = len(action2idx)
print(f"Vocab size: {VOCAB_SIZE}")


def prepare_sequences(df):
    seq    = df[feature_cols].values
    mapped = np.zeros_like(seq, dtype=np.int64)
    for i in range(seq.shape[0]):
        for j in range(seq.shape[1]):
            v = seq[i, j]
            if pd.isna(v):
                mapped[i, j] = 0          # PAD
            else:
                mapped[i, j] = action2idx.get(v, 1)  # UNK fallback
    return torch.tensor(mapped, dtype=torch.long)


X_tr_seq   = prepare_sequences(X_train)
X_va_seq   = prepare_sequences(X_val)
X_test_seq = prepare_sequences(X_test)

# ─────────────────────────────────────────────
# FIX 2: Build label maps from UNION of train+val
# to avoid NaN in val after mapping
# ─────────────────────────────────────────────
label_cols = [c for c in Y_train.columns if c != "id"]
label_maps = {}
num_classes = []

for col in label_cols:
    uniq    = sorted(set(Y_train[col].unique()) | set(Y_val[col].unique()))
    mapping = {v: i for i, v in enumerate(uniq)}
    label_maps[col] = mapping
    num_classes.append(len(uniq))
    Y_train[col] = Y_train[col].map(mapping)
    Y_val[col]   = Y_val[col].map(mapping)

print(f"Num classes per target: {num_classes}")

y_tr = torch.tensor(Y_train[label_cols].values, dtype=torch.long)
y_va = torch.tensor(Y_val[label_cols].values,   dtype=torch.long)


class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


train_loader = DataLoader(SeqDataset(X_tr_seq, y_tr),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(SeqDataset(X_va_seq, y_va),
                          batch_size=BATCH_SIZE, num_workers=0)


# ─────────────────────────────────────────────
# FIX 3: Residual Dilated Block (avoids vanishing gradient)
# ─────────────────────────────────────────────
class ResidualDilatedBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3,
                               padding=dilation, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3,
                               padding=dilation, dilation=dilation)
        self.bn1   = nn.BatchNorm1d(channels)
        self.bn2   = nn.BatchNorm1d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + residual)   # skip connection


# ─────────────────────────────────────────────
# FIX 4: Enhanced architecture
#   - Positional encoding (sequence order matters!)
#   - Residual dilated blocks
#   - Both max AND mean pooling (2x richer representation)
#   - Label correlation via shared trunk → task-specific heads
# ─────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, d_model, L) → add positional info
        return x + self.pe[:, :x.size(2), :].transpose(1, 2)


class EnhancedDilatedCNN(nn.Module):
    def __init__(self, vocab_size, num_classes, seq_len):
        super().__init__()
        EMBED = 128
        CH    = 256

        self.embedding = nn.Embedding(vocab_size, EMBED, padding_idx=0)
        self.pos_enc   = PositionalEncoding(EMBED, max_len=seq_len + 1)

        # Project embedding → channel dim
        self.input_proj = nn.Sequential(
            nn.Conv1d(EMBED, CH, kernel_size=1),
            nn.BatchNorm1d(CH),
            nn.ReLU()
        )

        # Stacked residual dilated blocks
        self.blocks = nn.Sequential(
            ResidualDilatedBlock(CH, dilation=1),
            ResidualDilatedBlock(CH, dilation=2),
            ResidualDilatedBlock(CH, dilation=4),
            ResidualDilatedBlock(CH, dilation=8),
            ResidualDilatedBlock(CH, dilation=16),
            ResidualDilatedBlock(CH, dilation=1),   # reset receptive field
        )

        self.max_pool  = nn.AdaptiveMaxPool1d(1)
        self.avg_pool  = nn.AdaptiveAvgPool1d(1)
        self.dropout   = nn.Dropout(0.4)

        # Shared trunk for label correlation
        trunk_in = CH * 2   # max + avg concat
        self.trunk = nn.Sequential(
            nn.Linear(trunk_in, CH),
            nn.LayerNorm(CH),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Per-task heads
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(CH, 128),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(128, n)
            )
            for n in num_classes
        ])

    def forward(self, x):
        # x: (B, L)
        emb = self.embedding(x)           # (B, L, EMBED)
        emb = emb.transpose(1, 2)         # (B, EMBED, L)
        emb = self.pos_enc(emb)           # add positional encoding

        feat = self.input_proj(emb)       # (B, CH, L)
        feat = self.blocks(feat)          # (B, CH, L)

        # FIX: dual pooling captures both salient and average signal
        max_f = self.max_pool(feat).squeeze(-1)   # (B, CH)
        avg_f = self.avg_pool(feat).squeeze(-1)   # (B, CH)
        pooled = torch.cat([max_f, avg_f], dim=1) # (B, CH*2)
        pooled = self.dropout(pooled)

        shared = self.trunk(pooled)               # (B, CH)
        return [head(shared) for head in self.heads]


model = EnhancedDilatedCNN(VOCAB_SIZE, num_classes, SEQ_LEN).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# FIX 5: Cosine annealing – avoids getting stuck in local minima
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)   # FIX 6: label smoothing

# ─────────────────────────────────────────────
# Training loop with early stopping
# ─────────────────────────────────────────────
best_exact = 0.0
patience   = 10
no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for xb, yb in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(xb)
        loss = sum(criterion(outputs[i], yb[:, i]) for i in range(len(num_classes)))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # FIX 7: grad clip
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()

    # ── Validation ──
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            outputs = model(xb)
            preds = torch.stack([o.argmax(1).cpu() for o in outputs], dim=1)
            all_preds.append(preds)
            all_true.append(yb)

    all_preds = torch.cat(all_preds)
    all_true  = torch.cat(all_true)

    per_col_acc = [(all_preds[:, i] == all_true[:, i]).float().mean().item()
                   for i in range(len(num_classes))]
    exact_match = (all_preds == all_true).all(dim=1).float().mean().item()

    acc_str = " | ".join([f"attr_{i+1}:{a:.3f}" for i, a in enumerate(per_col_acc)])
    print(f"Epoch {epoch+1:3d} | Loss: {total_loss:.2f} | "
          f"ExactMatch: {exact_match:.4f} | {acc_str}")

    if exact_match > best_exact:
        best_exact = exact_match
        no_improve = 0
        torch.save(model.state_dict(), "/tmp/best_model.pt")
        print(f"  ✓ Best model saved (exact={best_exact:.4f})")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

# ─────────────────────────────────────────────
# Load best model and generate submission
# ─────────────────────────────────────────────
model.load_state_dict(torch.load("/tmp/best_model.pt", map_location=DEVICE))
model.eval()

all_test_preds = []
test_loader = DataLoader(
    SeqDataset(X_test_seq, torch.zeros(len(X_test_seq), len(num_classes), dtype=torch.long)),
    batch_size=BATCH_SIZE
)

with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(DEVICE)
        outputs = model(xb)
        preds = torch.stack([o.argmax(1).cpu() for o in outputs], dim=1)
        all_test_preds.append(preds)

all_test_preds = torch.cat(all_test_preds).numpy()

# Reverse label map
for i, col in enumerate(label_cols):
    rev = {v: k for k, v in label_maps[col].items()}
    all_test_preds[:, i] = np.vectorize(rev.get)(all_test_preds[:, i])

submission = pd.DataFrame(all_test_preds.astype(np.uint16), columns=label_cols)
submission.insert(0, "id", X_test["id"])
submission.to_csv("submission_enhanced.csv", index=False)
print(f"\nDone. Best val exact match: {best_exact:.4f}")
print("Submission saved to submission_enhanced.csv")


Using device: cuda
Vocab size: 256
Num classes per target: [12, 31, 99, 12, 31, 99]
Model parameters: 2,800,668


Epoch   1 | Loss: 3331.52 | ExactMatch: 0.0000 | attr_1:0.905 | attr_2:0.397 | attr_3:0.008 | attr_4:0.913 | attr_5:0.397 | attr_6:0.010


Epoch   2 | Loss: 2956.57 | ExactMatch: 0.0000 | attr_1:0.980 | attr_2:0.498 | attr_3:0.009 | attr_4:0.980 | attr_5:0.470 | attr_6:0.010


Epoch   3 | Loss: 2683.06 | ExactMatch: 0.0001 | attr_1:0.980 | attr_2:0.870 | attr_3:0.009 | attr_4:0.980 | attr_5:0.760 | attr_6:0.012
  ✓ Best model saved (exact=0.0001)


Epoch   4 | Loss: 2427.27 | ExactMatch: 0.0003 | attr_1:0.984 | attr_2:0.995 | attr_3:0.011 | attr_4:0.972 | attr_5:0.974 | attr_6:0.009
  ✓ Best model saved (exact=0.0003)


...

Epoch  46 | Loss: 776.78 | ExactMatch: 0.9894 | attr_1:0.999 | attr_2:1.000 | attr_3:0.996 | attr_4:0.997 | attr_5:0.999 | attr_6:0.995


Epoch  47 | Loss: 776.35 | ExactMatch: 0.9893 | attr_1:0.999 | attr_2:1.000 | attr_3:0.996 | attr_4:0.997 | attr_5:0.999 | attr_6:0.995


Epoch  48 | Loss: 773.52 | ExactMatch: 0.9893 | attr_1:0.999 | attr_2:1.000 | attr_3:0.996 | attr_4:0.997 | attr_5:0.999 | attr_6:0.995


Epoch  49 | Loss: 773.72 | ExactMatch: 0.9892 | attr_1:0.999 | attr_2:1.000 | attr_3:0.996 | attr_4:0.997 | attr_5:0.999 | attr_6:0.995


Epoch  50 | Loss: 772.58 | ExactMatch: 0.9901 | attr_1:0.999 | attr_2:1.000 | attr_3:0.996 | attr_4:0.997 | attr_5:0.999 | attr_6:0.996
  ✓ Best model saved (exact=0.9901)

Done. Best val exact match: 0.9901
Submission saved to submission_enhanced.csv
